In [1]:
import joblib
import mlflow
import pandas as pd
import os
import sys
from interpret import show
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
sys.path.append(os.path.dirname(os.getcwd()))
from models.interface import model_interface
from data.DataLoader import DataLoader

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Retail Project")

<Experiment: artifact_location='./mlruns/1', experiment_id='1', lifecycle_stage='active', name='Retail Project', tags={}>

In [2]:
def find_mlruns_parent():
	"""Search upward from current directory for a folder containing 'mlruns'."""
	current = os.path.abspath(".")
	while True:
		if os.path.exists(os.path.join(current, "mlruns")):
			return current
		parent = os.path.dirname(current)
		if parent == current:
			break
		current = parent
	raise FileNotFoundError("Could not find 'mlruns' directory. Please specify the path manually.")

mlruns_parent = find_mlruns_parent()

In [3]:
model_names = [
    "Explainable Boosting Machine",
    "ElasticNet Regression",
    "XGBoost Regression",
]

latest_runs = {}

for model_name in model_names:
        runs = mlflow.search_runs(
                experiment_names=["Retail Project"],
                filter_string=f"tags.task = 'regression' AND tags.model_name = '{model_name}' AND attribute.status = 'FINISHED'",
                order_by=["attribute.start_time DESC"],
                max_results=1,
        )

        if runs.empty:
                print(f"No run found for {model_name}")
                continue
        latest_runs[model_name] = runs.iloc[0]
        print(latest_runs[model_name])

run_id                                            fd0679905e914f75b9cf2880b3fb04fd
experiment_id                                                                    1
status                                                                    FINISHED
artifact_uri                     ./mlruns/1/fd0679905e914f75b9cf2880b3fb04fd/ar...
start_time                                        2026-09-02 14:11:39.922000+00:00
end_time                                          2026-09-02 14:12:41.942000+00:00
metrics.train_rmse                                                      105.103703
metrics.train_mse                                                     11046.788356
metrics.test_mse                                                      12165.647798
metrics.test_rmse                                                       110.297995
metrics.test_mae                                                         38.015703
metrics.test_r2                                                           0.584827
metr

In [4]:
models = {}
metrics_summary = []
for model_name, run in latest_runs.items():
    run_id = run.run_id
    
    # 1. Load the model
    model_file = os.path.join(mlruns_parent, "mlruns", "1", run_id, "artifacts", "model", "model.joblib")
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model file not found: {model_file}")
    model = joblib.load(model_file)
    
    # 2. Get run metadata from MLflow
    mlflow_run = mlflow.get_run(run_id)
    
    # 3. Extract metrics (e.g., rmse, mae, r2)
    metrics = mlflow_run.data.metrics
    
    # 4. Extract hyperparameters
    params = mlflow_run.data.params
    
    # 5. Store everything in a structured dict
    models[model_name] = {
        'model': model,
        'metrics': metrics,
        'params': params
    }
    
    # 6. Collect for a summary DataFrame (optional)
    metrics_summary.append({
        'model': model_name,
        **metrics,
        **params
    })

# Create a summary DataFrame for easy comparison
summary_df = pd.DataFrame(metrics_summary)
summary_df.to_csv("result_data/regression_model_info.csv", index=False)

In [5]:
ebm_model = models[model_names[0]]['model'].model
ebm_global = ebm_model.explain_global()
show(ebm_global)

<!-- http://127.0.0.1:7001/140293281347488/ -->

In [6]:
data = ebm_global.data()

# Print feature names and their importance scores
for name, score in zip(data['names'], data['scores']):
    print(f"{name}: {score:.4f}")

Sales: 199.1447
Quantity: 2.5075
Discount: 46.8305
Shipping Cost: 7.6552
Sales_per_Quantity: 8.3032
Discounted_Sales: 246.9372
Shipping_Cost_per_Unit: 8.5351
Order Priority_Critical: 0.2665
Order Priority_High: 0.7241
Order Priority_Low: 0.6319
Order Priority_Medium: 0.1215
Market_APAC: 2.2352
Market_Africa: 0.0257
Market_Canada: 0.0153
Market_EMEA: 0.0803
Market_EU: 1.0007
Market_LATAM: 0.9483
Market_US: 3.1893
Region_Africa: 0.0266
Region_Canada: 0.0095
Region_Caribbean: 0.1970
Region_Central: 0.2787
Region_Central Asia: 0.6100
Region_EMEA: 0.0833
Region_East: 0.0055
Region_North: 0.4285
Region_North Asia: 0.0149
Region_Oceania: 0.7860
Region_South: 0.5509
Region_Southeast Asia: 0.7358
Region_West: 0.1996
Segment_Consumer: 0.2302
Segment_Corporate: 0.0157
Segment_Home Office: 0.2735
Ship Mode_First Class: 0.1481
Ship Mode_Same Day: 0.4989
Ship Mode_Second Class: 0.6710
Ship Mode_Standard Class: 0.0560
Category_Furniture: 2.8779
Category_Office Supplies: 1.2411
Category_Technology: 2.

## ElesticNet Regression

In [7]:
elestic_model_w = models[model_names[1]]['model']
elestic_model = elestic_model_w.model

summary_df[summary_df["model"] == model_names[1]]

,model,train_mse,train_r2,test_mse,train_mae,train_rmse,test_r2,test_rmse,test_mae,target_column,...,interactions,l1_ratio,alpha,polynomial_degree,n_estimators,colsample_bytree,reg_alpha,max_depth,reg_lambda,subsample
1,ElasticNet Regression,796.009687,0.737878,994.588019,15.663107,28.213644,0.727695,31.53709,16.453243,Profit,...,NaN,0.5,0.5,2,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
feature_names = elestic_model_w.polynomial_features.get_feature_names_out()
coefs = dict(zip(feature_names, elestic_model.coef_))
sorted_coefs = sorted(coefs.items(), key=lambda x: abs(x[1]), reverse=True)

for name, coef in sorted_coefs:
    print(f"{name}: {coef:.4f}")

Discount Discounted_Sales: 8.5807
Sales Discount: -4.4581
Discounted_Sales Market_US: 2.6059
Sales Market_US: -2.2638
Discounted_Sales Market_Africa: 1.3881
Sales Market_Africa: -1.2048
Discounted_Sales Region_Central: 1.1008
Discounted_Sales Market_EMEA: 1.0416
Quantity Market_US: 1.0221
Discounted_Sales Market_APAC: 0.9893
Sales Region_Central: -0.9611
Discounted_Sales Region_Caribbean: 0.9593
Discounted_Sales Region_West: 0.9410
Quantity Discount: -0.9359
Sales Market_EMEA: -0.9152
Discounted_Sales Region_East: 0.8955
Discounted_Sales Region_South: 0.8947
Discounted_Sales Region_North: 0.8444
Sales Region_West: -0.8049
Discounted_Sales Sub-Category_Appliances: 0.7852
Sales Market_APAC: -0.7720
Sales Region_Caribbean: -0.7626
Sales Region_South: -0.7505
Sales Sub-Category_Appliances: -0.7404
Sales Region_East: -0.7223
Sales Region_North: -0.6984
Discount Shipping Cost: -0.6752
Discount Sales_per_Quantity: -0.6294
Quantity Market_EU: -0.5467
Quantity Market_APAC: -0.5296
Discounted_Sa

In [9]:
sorted_df = pd.DataFrame(sorted_coefs, columns=['Feature', 'Coefficient'])
sorted_df = sorted_df.reindex(sorted_df['Coefficient'].abs().sort_values(ascending=False).index).head(10)
sorted_df = sorted_df.sort_values('Coefficient', ascending=False)
sorted_df.to_csv("result_data/elestic_regression_feature_coefs.csv")

colors = ['#ff7f0e' if c > 0 else '#1f77b4' for c in sorted_df['Coefficient']]
fig = go.Figure()
fig.add_trace(go.Bar(
	y=sorted_df['Feature'],
	x=sorted_df['Coefficient'],
	orientation='h',
	marker_color=colors,
	text=[f"{c:.4f}" for c in sorted_df['Coefficient']],
	textposition='outside',
	hovertemplate='<b>%{y}</b><br>Coefficient: %{x:.4f}<br>Impact: %{text}<extra></extra>'
))
fig.update_layout(
	title={
		'text': 'Elestic Regression Feature Effects',
		'font': {'size': 20}
	},
	xaxis_title='Coefficient Value (Impact on Prediction)',
	yaxis_title='Features',
	height=600,
	width=1000,
	showlegend=False,
	bargap=0.3
)

fig.show()

In [10]:
xgb_model_w = models[model_names[2]]['model']
xgb_model = xgb_model_w.model
feature_names = xgb_model.feature_names_in_

# 1. Weight - how many times a feature is used to split
importance_weight = pd.Series(xgb_model.feature_importances_, index=feature_names).fillna(0)

# 2. Gain - average gain of splits using that feature (best for overall impact)
importance_gain = pd.Series(xgb_model.get_booster().get_score(importance_type='gain'), index=feature_names).fillna(0)
# ig_df = pd.DataFrame(importance_gain).T

# 3. Cover - average coverage of splits using that feature
importance_cover = pd.Series(xgb_model.get_booster().get_score(importance_type='cover'),
                             index=feature_names).fillna(0)

top_weight = importance_weight.sort_values().tail(15)
top_gain = importance_gain.sort_values().tail(15)
top_cover = importance_cover.sort_values().tail(15)
top_gain.to_csv("result_data/xgb_regression_feature_gain.csv")
top_cover.to_csv("result_data/xgb_regression_feature_cover.csv")

fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=('Weight', 'Gain', 'Cover'),
                    x_title='Importance Score')

fig.add_trace(go.Bar(x=top_weight.values, y=top_weight.index, 
                     orientation='h', name='Weight', marker_color='#1f77b4'),
              row=1, col=1)

fig.add_trace(go.Bar(x=top_gain.values, y=top_gain.index, 
                     orientation='h', name='Gain', marker_color='#ff7f0e'),
              row=2, col=1)

fig.add_trace(go.Bar(x=top_cover.values, y=top_cover.index, 
                     orientation='h', name='Cover', marker_color='#2ca02c'),
              row=3, col=1)

fig.update_layout(height=1000, width=600, showlegend=False, title_text="XGBoost Feature Importances Comparison")
fig.update_xaxes(title_text="Importance", row=1, col=1)
fig.update_xaxes(title_text="Importance", row=1, col=2)
fig.update_xaxes(title_text="Importance", row=1, col=3)

fig.show()

In [11]:
loader = DataLoader(
	excel_location="data/data.xlsx",
	classification_excel_location = "data/classification_data.xlsx",
	train_test_split_percentage=0.8
)

loader.load()
valid, log = loader.validate()
print(log)
if not valid:
	raise RuntimeError("Dataset validation failed.")


regression_train_df = loader.get_regression_train_dataframe()
regression_test_df = loader.get_regression_test_dataframe()

regression_test_df

--------------------- DataLoader Validation ---------------------
FIXES:
  [FIXED] Sheet 'DimGeography': column 'City' converted from 'object' to 'str'.
  [FIXED] Sheet 'DimGeography': column 'City ASCII' converted from 'object' to 'str'.

Validation: PASSED


,Order ID,Customer ID,Order Priority,Order Date,Market,Product ID,Sales,Quantity,Discount,Profit,...,Category,Sub-Category,Id,Key,Lat,Lng,Country,State,City,City ASCII
0,ES-2013-4511365,sn-20710,Medium,"Tuesday, October 29, 2013",EU,off-bi-10003708,42.570000,3,0.0,8.910000,...,Office Supplies,Binders,2648579.0,united kingdom scotland glasgow,55.86515,-4.25763,United Kingdom,Scotland,Glasgow,Glasgow
1,SF-2013-10000,nb-8580,Medium,"Tuesday, October 29, 2013",Africa,off-sti-10000937,23.610001,1,0.0,0.450000,...,Office Supplies,Supplies,964137.0,south africa gauteng pretoria,-25.74486,28.18783,South Africa,Gauteng,Pretoria,Pretoria
2,SF-2013-10000,nb-8580,Medium,"Tuesday, October 29, 2013",Africa,off-kle-10002118,10.260000,1,0.0,2.550000,...,Office Supplies,Supplies,964137.0,south africa gauteng pretoria,-25.74486,28.18783,South Africa,Gauteng,Pretoria,Pretoria
3,TU-2013-2640,cl-1890,Medium,"Tuesday, October 29, 2013",EMEA,off-bre-10001343,36.096001,1,0.6,-38.813999,...,Office Supplies,Appliances,298117.0,turkey van van,38.49457,43.38323,Turkey,Van,Van,Van
4,SF-2013-10000,nb-8580,Medium,"Tuesday, October 29, 2013",Africa,off-eld-10001832,96.900002,2,0.0,0.000000,...,Office Supplies,Storage,964137.0,south africa gauteng pretoria,-25.74486,28.18783,South Africa,Gauteng,Pretoria,Pretoria
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6326,CA-2011-109918,lr-17035,Medium,"Wednesday, September 7, 2011",US,off-pa-10000174,20.559999,2,0.0,9.663200,...,Office Supplies,Paper,5393015.0,united states california santa clara,37.35411,-121.95524,United States,California,Santa Clara,Santa Clara
6327,IN-2011-33253,pb-19210,High,"Wednesday, September 7, 2011",APAC,off-ar-10000780,34.560001,2,0.1,9.960000,...,Office Supplies,Art,2165798.0,australia victoria geelong,-38.14711,144.36069,Australia,Victoria,Geelong,Geelong
6328,IN-2011-33253,pb-19210,High,"Wednesday, September 7, 2011",APAC,off-fa-10002539,29.268000,2,0.1,1.248000,...,Office Supplies,Fasteners,2165798.0,australia victoria geelong,-38.14711,144.36069,Australia,Victoria,Geelong,Geelong
6329,MX-2011-149034,tc-20980,Medium,"Wednesday, September 7, 2011",LATAM,off-st-10000505,457.700012,5,0.0,64.000000,...,Office Supplies,Storage,3598529.0,guatemala guatemala chinautla,14.70289,-90.49983,Guatemala,Guatemala,Chinautla,Chinautla


In [12]:
elestic_result = models[model_names[1]]['model'].predict(regression_test_df)
pd.DataFrame({
        "y_true": elestic_result[0],
        "y_predict": elestic_result[1]
}).to_csv("result_data/elestic_regression_result.csv", index=False)

xgb_result = models[model_names[2]]['model'].predict(regression_test_df)
pd.DataFrame({
        "y_true": xgb_result[0],
        "y_predict": xgb_result[1]
}).to_csv("result_data/xgb_regression_result.csv", index=False)